# Trajectory Visualization
In this section, I want to recreate the 3D trajectory visualization with welleng utilization.
hopefully I understand the welleng survey object

In [23]:
import pandas as pd 
import welleng as we


In [24]:
TrajectoryFilename = "..\\Streamlit_App\\Data\\WellTrajectory_Examples.xlsx"

TrajectoryRaw_DF = pd.read_excel(TrajectoryFilename, sheet_name="Data")
TrajectoryRaw_DF


,MD (m),Inc (°),Azi (°),TVD (m),N/S (m),E/W (m),VSEC (m),DLS (°/30m),Map N (m),Map E (m)
0,0.00,0.00,0.0,0.00,-2495.31,-4817.34,0.00,0.000,9916092.75,567043.52
1,19.20,0.00,0.0,19.20,-2495.31,-4817.34,0.00,0.000,9916092.75,567043.52
2,30.00,0.68,212.0,30.00,-2495.37,-4817.37,0.06,1.887,9916092.70,567043.49
3,60.00,2.57,212.0,59.99,-2496.09,-4817.82,0.91,1.887,9916091.98,567043.04
4,90.00,4.45,212.0,89.93,-2497.65,-4818.79,2.74,1.887,9916090.42,567042.06
...,...,...,...,...,...,...,...,...,...,...
60,1650.00,55.00,204.0,1269.23,-3343.47,-5275.88,964.16,0.000,9915244.90,566585.14
61,1680.99,55.00,204.0,1287.00,-3366.66,-5286.20,989.48,0.000,9915221.72,566574.82
62,1710.00,55.00,204.0,1303.64,-3388.37,-5295.87,1013.18,0.000,9915200.02,566565.16
63,1720.98,55.00,204.0,1309.94,-3396.59,-5299.53,1022.16,0.000,9915191.80,566561.50


In [25]:

def LoadSurvey(TrajectoryDF):
    TrajectoryDF = TrajectoryDF.copy()
    ColumnUnitDict = {}
    ColumnRenameDict = {}

    for ColName in list(TrajectoryDF.columns):
        ColKeys=ColName.split(" (")[0]
        ColValues=ColName.split(" (")[1].split(")")[0]
        ColumnRenameDict[ColName] = ColKeys
        ColumnUnitDict[ColKeys] = ColValues
    TrajectoryDF.rename(columns = ColumnRenameDict, inplace = True)
    # TrajectoryDF[]
    # TrajectoryDF[]
    # TrajectoryDF[]
    surveyObj=we.survey.Survey(TrajectoryDF['MD'].tolist(),
            TrajectoryDF['Inc'].tolist(),
            TrajectoryDF['Azi'].tolist(),
            start_xyz=[0., 0., 0.],
            deg=True,
            unit="meters",
        )


    return surveyObj
TrajectorySurvey= LoadSurvey(TrajectoryRaw_DF)

TrajectorySurvey

In [26]:
we.visual._panel(TrajectorySurvey)

In [27]:
we.visual.figure(TrajectorySurvey, type='panel')

# Manual Trajectory Design
In this section, I want to create a function that generate trajectory survey based on minimumj curvature algoritm with manually input the relative xyz coordinates.


In [30]:
import pandas as pd 
import welleng as we
def emptyTargetPoint():
    TargetDF = pd.DataFrame(columns=[
        "X",
        "Y",
        "Z",
        "DLS"
    ])
    return TargetDF
TargetDF = emptyTargetPoint()
TargetDF

,X,Y,Z,DLS


In [31]:
X_input = 0
Y_input = 0
Z_input = 0
DLS_input = 3

TargetDF = TargetDF.append({'X': X_input, 'Y': Y_input, 'Z': Z_input, 'DLS': DLS_input}, 
                                ignore_index=True, 
                                sort=False)
X_input = 0
Y_input = 0
Z_input = 100
DLS_input = 3

TargetDF = TargetDF.append({'X': X_input, 'Y': Y_input, 'Z': Z_input, 'DLS': DLS_input}, 
                                ignore_index=True, 
                                sort=False)
X_input = 300
Y_input = 25
Z_input = 700
DLS_input = 3

TargetDF = TargetDF.append({'X': X_input, 'Y': Y_input, 'Z': Z_input, 'DLS': DLS_input}, 
                                ignore_index=True, 
                                sort=False)
X_input = 1000
Y_input = 300
Z_input = 1500
DLS_input = 2

TargetDF = TargetDF.append({'X': X_input, 'Y': Y_input, 'Z': Z_input, 'DLS': DLS_input}, 
                                ignore_index=True, 
                                sort=False)
TargetDF

C:\Users\irsya\AppData\Local\Temp\ipykernel_5884\1026411668.py:6: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

C:\Users\irsya\AppData\Local\Temp\ipykernel_5884\1026411668.py:14: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

C:\Users\irsya\AppData\Local\Temp\ipykernel_5884\1026411668.py:22: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

C:\Users\irsya\AppData\Local\Temp\ipykernel_5884\1026411668.py:30: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.



,X,Y,Z,DLS
0,0,0,0,3
1,0,0,100,3
2,300,25,700,3
3,1000,300,1500,2


In [4]:
def generateSurvey_v2(TargetDF, interpolation_step = 30, datum=15, init_inc=0, init_azi=0):
    # node_list = []
    connector_list = []
    i = 0
    for idx,row in TargetDF.iterrows():
        print([row['X'], row['Y'], row['Z'], row['DLS']])
        if i==0:
            pos1 = [row['X'], row['Y'], (row['Z'])]
            print("a")

        elif i==1:
            connector_list.append(we.connector.Connector(
                pos1=pos1,
                pos2=[row['X'], row['Y'], row['Z']],
                 md1 = -datum, inc1=init_inc, azi1=init_azi,dls_design=row['DLS']
            ))
            print("b")
        elif i>1:
            connector_list.append(we.connector.Connector(
                pos1=connector_list[-1].pos_target,
                pos2=[row['X'], row['Y'], row['Z']],
                vec1=connector_list[-1].vec_target,
                dls_design=row['DLS']

            ))
            print("c")
        i = i+1
    survey_example_2 = we.survey.from_connections(
        connector_list
        )
    return survey_example_2

In [73]:
def generateSurvey(TargetDF, interpolation_step = 30, datum=15, init_inc=0, init_azi=0):
    # node_list = []
    connector_list = []
    i = 0
    for idx,row in TargetDF.iterrows():
        # print([row['X'], row['Y'], row['Z'], row['DLS']])
        if i==0:

            node0 = (we.node.Node(pos=[row['X'], row['Y'], row['Z']], md=-datum, inc=init_inc, azi=init_azi))
        elif i==1:
            node = (we.node.Node(pos=[row['X'], row['Y'], row['Z']]))
            connector_list.append(
                we.connector.Connector(node0, node, dls_design=row['DLS'])
            )
            
        elif i>1:
            node = (we.node.Node(pos=[row['X'], row['Y'], row['Z']]))
            connector_list.append(
                we.connector.Connector(connector_list[-1].node_end, node,dls_design=row['DLS'])
            )
        i = i+1
    # print(connector_list)
    survey_example_2 = we.survey.from_connections(
        connector_list
        ).interpolate_survey(step=interpolation_step)
    return survey_example_2

In [35]:
generateSurvey(TargetDF, interpolation_step = 30, datum=15, init_inc=0, init_azi=0)

[0, 0, 0, 3]
[0, 0, 100, 3]
[300, 25, 700, 3]
[1000, 300, 1500, 2]
[<welleng.connector.Connector object at 0x000001F1FAD38460>, <welleng.connector.Connector object at 0x000001F1FAD38A90>, <welleng.connector.Connector object at 0x000001F1FAD38CD0>]


In [32]:
TargetList = []
DLSList = []
for idx,row in TargetDF.iterrows():
    TargetList.append(
        [float(row['X']), float(row['Y']), float(row['Z'])]
    )
    DLSList.append(float(row['DLS']))


In [34]:
TargetList

[[300.0, 25.0, 1500.0]]

In [35]:

# Push the points to the connect_points function to generate a survey
connections = we.connector.connect_points(
    TargetList,

    dls_design=3.,
    md_start=-15
    # step=30,

)
survey = we.survey.from_connections(connections)

ValueError: need at least one array to concatenate

In [75]:
we.survey.export_csv(survey, None)

,MD,INC (deg),AZI (deg),NORTHING (m),EASTING (m),TVDSS (m),DLS,TOOLFACE,BUILD RATE,TURN RATE
0,-15.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.0,0.000000,0.000000,0.000000
1,85.000000,0.000000,0.000000,0.000000,0.000000,-100.000000,0.0,0.000000,0.000000,0.000000
2,369.977022,28.497702,4.763642,69.181917,5.765160,-373.371638,3.0,4.763642,3.000000,0.501476
3,855.423957,28.497702,4.763642,300.000000,25.000000,-800.000000,0.0,0.000000,0.000000,0.000000
4,1245.587455,10.518648,184.763642,359.586932,29.965578,-1177.968250,3.0,180.000000,1.382425,13.840352
5,1573.123265,10.518648,184.763642,300.000000,25.000000,-1500.000000,0.0,0.000000,0.000000,0.000000


In [81]:
Survey_Obj = generateSurvey_v2(TargetDF)
we.visual.figure(survey, type='scatter3d')

[0, 0, 0, 3]
a
[0, 0, 100, 3]
b
[300, 25, 800, 3]
c
[300, 25, 1500, 3]
c


In [37]:
for idx,row in TargetDF.iterrows():
    A = [row['X'], row['Y'], (row['Z'])]
A

[1500, 1500, 2500]

In [40]:
A[-1]

2500

## Trajectory Progress Input


In [1]:
import pandas as pd 
import welleng as we


TrajectoryActual_Filename = "..\\Streamlit_App\\Data\\WellTrajectory_Examples_Actual.xlsx"
TrajectoryPlan_Filename = "..\\Streamlit_App\\Data\\WellTrajectory_Examples_Current.xlsx"

TrajectoryRaw_Actual = pd.read_excel(TrajectoryActual_Filename, sheet_name="Data")
TrajectoryRaw_Plan = pd.read_excel(TrajectoryPlan_Filename, sheet_name="Data")
# TrajectoryRaw_DF


def LoadSurvey(TrajectoryDF):
    TrajectoryDF = TrajectoryDF.copy()
    ColumnUnitDict = {}
    ColumnRenameDict = {}

    for ColName in list(TrajectoryDF.columns):
        ColKeys=ColName.split(" (")[0]
        ColValues=ColName.split(" (")[1].split(")")[0]
        ColumnRenameDict[ColName] = ColKeys
        ColumnUnitDict[ColKeys] = ColValues
    TrajectoryDF.rename(columns = ColumnRenameDict, inplace = True)
    # TrajectoryDF[]
    # TrajectoryDF[]
    # TrajectoryDF[]
    surveyObj=we.survey.Survey(TrajectoryDF['MD'].tolist(),
            TrajectoryDF['Inc'].tolist(),
            TrajectoryDF['Azi'].tolist(),
            start_xyz=[0., 0., 0.],
            deg=True,
            unit="meters",
        )


    return surveyObj
TrajectorySurvey_Actual= LoadSurvey(pd.read_excel(TrajectoryActual_Filename, sheet_name="Data"))
TrajectorySurvey_Plan= LoadSurvey(pd.read_excel(TrajectoryPlan_Filename, sheet_name="Data"))
TrajectorySurvey_Plan

In [2]:
# we.survey.export_csv(TrajectorySurvey)
# TrajectorySurvey = TrajectorySurvey.interpolate_survey(step=30)
TrajectorySurvey_Actual= LoadSurvey(pd.read_excel(TrajectoryActual_Filename, sheet_name="Data"))
TrajectorySurvey_Plan= LoadSurvey(pd.read_excel(TrajectoryPlan_Filename, sheet_name="Data"))
TrajectorySurvey_Plan = TrajectorySurvey_Plan.interpolate_survey(step=10)
TrajectorySurvey_Actual = TrajectorySurvey_Actual.interpolate_survey(step=10)
TrajectoryDF_Plan = (we.survey.export_csv(TrajectorySurvey_Plan, None))
TrajectoryDF_Actual = (we.survey.export_csv(TrajectorySurvey_Actual, None))
TrajectoryDF_Actual

,MD,INC (deg),AZI (deg),NORTHING (m),EASTING (m),TVDSS (m),DLS,TOOLFACE,BUILD RATE,TURN RATE
0,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000
1,30.480000,0.000000,0.000000,0.000000,0.000000,-30.480000,0.000000,0.000000,0.000000,0.000000
2,40.000000,0.937009,2.875000,0.077745,0.003904,-39.999576,2.952761,2.875000,2.952761,9.059874
3,50.000000,1.921263,2.875000,0.326833,0.016414,-49.996342,2.952761,0.000000,2.952761,0.000000
4,60.000000,2.905517,2.875000,0.747390,0.037534,-59.987349,2.952761,0.000000,2.952761,0.000000
...,...,...,...,...,...,...,...,...,...,...
66,980.000000,34.425871,73.012473,341.582983,121.621247,-885.015821,3.335357,59.785752,1.713892,5.098688
67,990.000000,35.019996,74.662339,343.167844,127.091900,-893.235130,3.335357,58.379031,1.782373,4.949599
68,1000.000000,35.635789,76.263228,344.618604,132.689022,-901.393830,3.335357,57.022930,1.847380,4.802667
69,1010.000000,36.272131,77.816051,345.934719,138.410504,-909.488850,3.335357,55.716788,1.909026,4.658468


In [3]:
def generateSurvey(TargetDF, interpolation_step = 30, datum=15, init_inc=0, init_azi=0):
    # node_list = []
    connector_list = []
    i = 0
    for idx,row in TargetDF.iterrows():
        # print([row['X'], row['Y'], row['Z'], row['DLS']])
        if i==0:

            node0 = (we.node.Node(pos=[row['X'], row['Y'], row['Z']], md=-datum, inc=init_inc, azi=init_azi))
        elif i==1:
            node = (we.node.Node(pos=[row['X'], row['Y'], row['Z']]))
            connector_list.append(
                we.connector.Connector(node0, node, dls_design=row['DLS'])
            )
            
        elif i>1:
            node = (we.node.Node(pos=[row['X'], row['Y'], row['Z']]))
            connector_list.append(
                we.connector.Connector(connector_list[-1].node_end, node,dls_design=row['DLS'])
            )
        i = i+1
    # print(connector_list)
    survey_example_2 = we.survey.from_connections(
        connector_list
        ).interpolate_survey(step=interpolation_step)
    return survey_example_2
generateSurvey(TargetDF, interpolation_step = 30, datum=15, init_inc=0, init_azi=0)

NameError: name 'TargetDF' is not defined

In [4]:
import plotly.express as px
import plotly.graph_objects as go
# TrajectoryDF = TrajectoryDF_Plan
fig = go.Figure(data=[
    go.Scatter3d(x=TrajectoryDF_Actual['NORTHING (m)'], y=TrajectoryDF_Actual['EASTING (m)'], z=TrajectoryDF_Actual['TVDSS (m)'],
                                   mode='lines'),
    go.Scatter3d(x=TrajectoryDF_Plan['NORTHING (m)'], y=TrajectoryDF_Plan['EASTING (m)'], z=TrajectoryDF_Plan['TVDSS (m)'],
                                   mode='lines'),
                                   ]
                                   )
fig.update_layout(
    scene = dict(
        xaxis = dict(nticks=4, range=[0,500],),
        yaxis = dict(nticks=4, range=[0,500],),
        zaxis = dict(nticks=4, range=[-1000,0],),),
    width=700,
    margin=dict(r=20, l=10, b=10, t=10))


    
fig.show()

# tight layout
# fig.update_layout(margin=dict(l=0, r=0, b=0, t=0))

In [71]:
TrajectorySurvey_Actual= LoadSurvey(pd.read_excel(TrajectoryActual_Filename, sheet_name="Data"))
TrajectorySurvey_Plan= LoadSurvey(pd.read_excel(TrajectoryPlan_Filename, sheet_name="Data"))
TrajectorySurvey_Plan = TrajectorySurvey_Plan.interpolate_survey(step=10)
TrajectorySurvey_Actual = TrajectorySurvey_Actual.interpolate_survey(step=10)
TrajectoryDF_Plan = (we.survey.export_csv(TrajectorySurvey_Plan, None))
TrajectoryDF_Actual = (we.survey.export_csv(TrajectorySurvey_Actual, None))
display(TrajectoryDF_Actual)
# TrajectoryDF_Actual
x_node = 346.672055
y_node = 141.964968
z_node = 914.400004
target_node = we.node.Node(pos=[x_node, y_node, z_node])
TrajectoryDF_Plan2 = TrajectorySurvey_Plan.project_to_target(target_node,dls_design=1, dls=None)

TrajectoryDF_Plan2 = TrajectoryDF_Plan2.interpolate_survey(step=10)
TrajectoryDF_Plan2 = (we.survey.export_csv(TrajectoryDF_Plan2, None))
TrajectoryDF_Plan

,MD,INC (deg),AZI (deg),NORTHING (m),EASTING (m),TVDSS (m),DLS,TOOLFACE,BUILD RATE,TURN RATE
0,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000
1,30.480000,0.000000,0.000000,0.000000,0.000000,-30.480000,0.000000,0.000000,0.000000,0.000000
2,40.000000,0.937009,2.875000,0.077745,0.003904,-39.999576,2.952761,2.875000,2.952761,9.059874
3,50.000000,1.921263,2.875000,0.326833,0.016414,-49.996342,2.952761,0.000000,2.952761,0.000000
4,60.000000,2.905517,2.875000,0.747390,0.037534,-59.987349,2.952761,0.000000,2.952761,0.000000
...,...,...,...,...,...,...,...,...,...,...
66,980.000000,34.425871,73.012473,341.582983,121.621247,-885.015821,3.335357,59.785752,1.713892,5.098688
67,990.000000,35.019996,74.662339,343.167844,127.091900,-893.235130,3.335357,58.379031,1.782373,4.949599
68,1000.000000,35.635789,76.263228,344.618604,132.689022,-901.393830,3.335357,57.022930,1.847380,4.802667
69,1010.000000,36.272131,77.816051,345.934719,138.410504,-909.488850,3.335357,55.716788,1.909026,4.658468


,MD,INC (deg),AZI (deg),NORTHING (m),EASTING (m),TVDSS (m),DLS,TOOLFACE,BUILD RATE,TURN RATE
0,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000
1,30.480000,0.000000,0.000000,0.000000,0.000000,-30.480000,0.000000,0.000000,0.000000,0.000000
2,40.000000,0.937009,2.875000,0.077745,0.003904,-39.999576,2.952761,2.875000,2.952761,9.059874
3,50.000000,1.921263,2.875000,0.326833,0.016414,-49.996342,2.952761,0.000000,2.952761,0.000000
4,60.000000,2.905517,2.875000,0.747390,0.037534,-59.987349,2.952761,0.000000,2.952761,0.000000
5,70.000000,3.889770,2.875000,1.339289,0.067260,-69.969649,2.952761,0.000000,2.952761,0.000000
6,80.000000,4.874024,2.875000,2.102358,0.105581,-79.940296,2.952761,0.000000,2.952761,0.000000
7,90.000000,5.858277,2.875000,3.036371,0.152488,-89.896348,2.952761,0.000000,2.952761,0.000000
8,100.000000,6.842531,2.875000,4.141052,0.207965,-99.834866,2.952761,0.000000,2.952761,0.000000
9,110.000000,7.826784,2.875000,5.416075,0.271997,-109.752918,2.952761,0.000000,2.952761,0.000000


In [27]:
fig = go.Figure(data=[
    go.Scatter3d(x=TrajectoryDF_Actual['NORTHING (m)'], y=TrajectoryDF_Actual['EASTING (m)'], z=TrajectoryDF_Actual['TVDSS (m)'],
                                   mode='markers+lines'),
    go.Scatter3d(x=TrajectoryDF_Plan['NORTHING (m)'], y=TrajectoryDF_Plan['EASTING (m)'], z=TrajectoryDF_Plan['TVDSS (m)'],
                                   mode='markers+lines'),
    go.Scatter3d(x=TrajectoryDF_Plan2['NORTHING (m)'], y=TrajectoryDF_Plan2['EASTING (m)'], z=TrajectoryDF_Plan2['TVDSS (m)'],
                                   mode='markers+lines'),
                                   ]
                                   )
fig.show()


In [87]:
new_target = [346.672055,141.964968,914.400004]
Desire_DLS = 2
TrajectoryDF_Test = TrajectoryDF_Plan.copy()
TrajectoryDF_Test.loc[TrajectoryDF_Test['DLS'] <=0, 'DLS']=0.0001
TargetDF_new = pd.DataFrame(
    {
        "X":list(TrajectoryDF_Test['EASTING (m)']) + [new_target[1]],
        "Y":list(TrajectoryDF_Test['NORTHING (m)'])+ [new_target[0]],
        "Z":list(-TrajectoryDF_Test['TVDSS (m)'])+ [new_target[2]],
        "DLS":list(TrajectoryDF_Test['DLS'])+ [Desire_DLS],
    }
)
out_survey = generateSurvey(TargetDF_new, interpolation_step = 30, datum=15, init_inc=0, init_azi=0)
out_df = we.survey.export_csv(out_survey, None)



fig = go.Figure(data=[
    go.Scatter3d(y=TrajectoryDF_Actual['NORTHING (m)'], x=TrajectoryDF_Actual['EASTING (m)'], z=TrajectoryDF_Actual['TVDSS (m)'],
                                   mode='markers+lines'),
    go.Scatter3d(y=TrajectoryDF_Plan['NORTHING (m)'], x=TrajectoryDF_Plan['EASTING (m)'], z=TrajectoryDF_Plan['TVDSS (m)'],
                                   mode='markers+lines'),
    go.Scatter3d(y=out_df['NORTHING (m)'], x=out_df['EASTING (m)'], z=out_df['TVDSS (m)'],
                                   mode='markers+lines'),
                                   ]
                                   )
fig.show()

In [78]:
TargetDF_new

,X,Y,Z,DLS
0,0.000000,0.000000,0.000000,0.000100
1,0.000000,0.000000,30.480000,0.000100
2,0.003904,0.077745,39.999576,2.952761
3,0.016414,0.326833,49.996342,2.952761
4,0.037534,0.747390,59.987349,2.952761
5,0.067260,1.339289,69.969649,2.952761
6,0.105581,2.102358,79.940296,2.952761
7,0.152488,3.036371,89.896348,2.952761
8,0.207965,4.141052,99.834866,2.952761
9,0.271997,5.416075,109.752918,2.952761


In [81]:
out_df

,MD,INC (deg),AZI (deg),NORTHING (m),EASTING (m),TVDSS (m),DLS,TOOLFACE,BUILD RATE,TURN RATE
0,-15.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000
1,15.480000,0.000000,0.000000,0.000000,0.000000,-30.480000,0.000000,0.000000,0.000000,0.000000
2,264.554018,24.515199,87.125000,2.632136,52.411726,-272.023471,2.952759,87.125000,2.952761,10.493869
3,441.468472,24.515199,87.125000,6.314070,125.727262,-432.989305,0.000000,0.000000,0.000000,0.000000
4,494.965017,29.149037,81.564501,8.783737,149.716642,-480.720606,2.952740,-30.950471,2.598582,-3.118238
5,494.999938,29.149037,81.564501,8.786233,149.733468,-480.751105,0.000000,0.000000,0.000000,0.000000
6,514.164970,30.861008,79.947612,10.329353,159.191663,-497.347079,2.952740,-25.983892,2.679836,-2.530999
7,514.999935,30.861008,79.947612,10.404113,159.613389,-498.063825,0.000000,0.000000,0.000000,0.000000
8,519.969362,31.450048,79.432041,10.864353,162.142823,-502.316480,3.903554,-24.583503,3.555979,-3.112457
9,782.774513,29.058689,44.554946,69.373796,275.199914,-731.062591,2.000000,-112.695039,0.272981,-3.981326
